In [1]:
# Section 1 - Imports & Setup
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# Section 2 - Load Both CSV Files
df1 = pd.read_csv("data/A_Z_medicines_dataset_of_india.csv")
df2 = pd.read_csv("data/medDataset_processed.csv")

print("=" * 50)
print("📊 Dataset 1 - A_Z Medicines")
print("=" * 50)
print(f"Rows: {len(df1)}")
print(f"Columns: {list(df1.columns)}")
print(df1.head(3))

print("\n" + "=" * 50)
print("📊 Dataset 2 - Processed Medicine Dataset")
print("=" * 50)
print(f"Rows: {len(df2)}")
print(f"Columns: {list(df2.columns)}")
print(df2.head(3))

📊 Dataset 1 - A_Z Medicines
Rows: 253973
Columns: ['id', 'name', 'price(₹)', 'Is_discontinued', 'manufacturer_name', 'type', 'pack_size_label', 'short_composition1', 'short_composition2']
   id                      name  price(₹)  Is_discontinued  \
0   1  Augmentin 625 Duo Tablet    223.42            False   
1   2       Azithral 500 Tablet    132.36            False   
2   3          Ascoril LS Syrup    118.00            False   

                      manufacturer_name       type         pack_size_label  \
0  Glaxo SmithKline Pharmaceuticals Ltd  allopathy     strip of 10 tablets   
1           Alembic Pharmaceuticals Ltd  allopathy      strip of 5 tablets   
2          Glenmark Pharmaceuticals Ltd  allopathy  bottle of 100 ml Syrup   

      short_composition1          short_composition2  
0  Amoxycillin  (500mg)      Clavulanic Acid (125mg)  
1   Azithromycin (500mg)                         NaN  
2   Ambroxol (30mg/5ml)    Levosalbutamol (1mg/5ml)   

📊 Dataset 2 - Processed Medic

In [3]:
# Section 3 - Clean & Combine Both Datasets

# --- Clean Dataset 1 (Medicine Info) ---
df1_clean = df1[[
    'name', 
    'price(₹)', 
    'manufacturer_name', 
    'type', 
    'pack_size_label',
    'short_composition1', 
    'short_composition2'
]].copy()

# Fill empty values
df1_clean.fillna("Not available", inplace=True)

# Convert each medicine row into a readable sentence
def medicine_to_text(row):
    return f"""Medicine: {row['name']}
Price: ₹{row['price(₹)']}
Manufacturer: {row['manufacturer_name']}
Type: {row['type']}
Pack Size: {row['pack_size_label']}
Composition: {row['short_composition1']}, {row['short_composition2']}"""

df1_clean['text'] = df1_clean.apply(medicine_to_text, axis=1)
print("✅ Dataset 1 cleaned!")
print(f"Sample:\n{df1_clean['text'][0]}")
print(f"\nTotal medicines: {len(df1_clean)}")

✅ Dataset 1 cleaned!
Sample:
Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Acid (125mg)

Total medicines: 253973


In [4]:
# Section 4 - Clean Dataset 2 (Medical Q&A)

# Drop rows where Question or Answer is empty
df2_clean = df2[['Question', 'Answer']].dropna()

# Convert Q&A into readable text
def qa_to_text(row):
    return f"""Question: {row['Question']}
Answer: {row['Answer']}"""

df2_clean['text'] = df2_clean.apply(qa_to_text, axis=1)
print("✅ Dataset 2 cleaned!")
print(f"Sample:\n{df2_clean['text'][0]}")
print(f"\nTotal Q&A pairs: {len(df2_clean)}")

✅ Dataset 2 cleaned!
Sample:
Question: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?
Answer: LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.

Total Q&A pairs: 16407


In [5]:
# Section 5 - Combine Both Datasets into One

all_texts = pd.concat([
    df1_clean[['text']],
    df2_clean[['text']]
], ignore_index=True)

print(f"✅ Total combined documents: {len(all_texts)}")
print(f"\n📊 Breakdown:")
print(f"   Medicine info entries : {len(df1_clean)}")
print(f"   Medical Q&A entries   : {len(df2_clean)}")
print(f"   Total                 : {len(all_texts)}")

✅ Total combined documents: 270380

📊 Breakdown:
   Medicine info entries : 253973
   Medical Q&A entries   : 16407
   Total                 : 270380


In [6]:
# Section 6 - Convert to LangChain Documents
from langchain_core.documents import Document

documents = []
for idx, row in all_texts.iterrows():
    doc = Document(
        page_content=row['text'],
        metadata={"row_id": idx}
    )
    documents.append(doc)

print(f"✅ Total LangChain documents created: {len(documents)}")
print(f"\nSample document:")
print(f"Content: {documents[0].page_content[:200]}")
print(f"Metadata: {documents[0].metadata}")

✅ Total LangChain documents created: 270380

Sample document:
Content: Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Aci
Metadata: {'row_id': 0}


In [7]:
# Section 7 - Split Documents into Chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks = splitter.split_documents(documents)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\nSample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

✅ Total chunks created: 359052

Sample chunk:
Content: Medicine: Augmentin 625 Duo Tablet
Price: ₹223.42
Manufacturer: Glaxo SmithKline Pharmaceuticals Ltd
Type: allopathy
Pack Size: strip of 10 tablets
Composition: Amoxycillin  (500mg) ,   Clavulanic Acid (125mg)
Metadata: {'row_id': 0}


In [8]:
# Section 8B - LOAD vectorstore (run this every restart)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embeddings
)

print(f"✅ Vectorstore loaded!")
print(f"📊 Total documents: {vectorstore._collection.count()}")

C:\Users\alpha\AppData\Local\Temp\ipykernel_21392\2097078005.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
C:\Users\alpha\AppData\Local\Temp\ipykernel_21392\2097078005.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


✅ Vectorstore loaded!
📊 Total documents: 361052


In [15]:
# Section 9 - Q&A Chain
from langchain_community.llms import Ollama

# Load LLM
llm = Ollama(model="mistral", temperature=0.3)
print("✅ Mistral LLM loaded!")

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)
print("✅ Retriever ready!")

# Q&A Function
def ask(question):
    docs = retriever.invoke(question)
    context = "\n".join([d.page_content for d in docs])
    
    full_prompt = f"""You are a helpful medical assistant specialized in Indian medicines.
Use the following context to answer the question.
If not found, say "I couldn't find this in my database."
Always remind users to consult a real doctor.

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(full_prompt)
    return response

print("✅ QA Chain ready!")

# Quick test
print("\n🧪 Testing with Paracetamol...")
result = ask("What is Paracetamol used for?")
print(f"Answer: {result}")

✅ Mistral LLM loaded!
✅ Retriever ready!
✅ QA Chain ready!

🧪 Testing with Paracetamol...
Answer:  Paracetamol, as found in the medicines listed, is primarily used as a pain reliever and fever reducer. It's often used to treat symptoms of headaches, muscle aches, toothaches, and fevers. However, it's important to consult a real doctor for proper diagnosis and dosage instructions before using any medication.


In [ ]:
# Section 10 - COOL Gradio UI
import gradio as gr

def ask_assistant(question, history):
    if not question.strip():
        return history, ""
    response = ask(question)
    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": response}
    ]
    return history, ""

def clear_chat():
    return [], ""

def quick_ask(question, history):
    return ask_assistant(question, history)

css = """
body {
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e) !important;
}
.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}
.header-text {
    text-align: center;
    padding: 20px;
    background: linear-gradient(90deg, #00c6ff, #0072ff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.5em !important;
    font-weight: bold;
}
.subtitle {
    text-align: center;
    color: #a0aec0;
    font-size: 1.1em;
}
.warning-box {
    background: rgba(255, 100, 100, 0.1);
    border: 1px solid #ff6464;
    border-radius: 10px;
    padding: 10px;
    text-align: center;
    color: #ff6464;
}
.stat-box {
    background: rgba(0, 198, 255, 0.1);
    border: 1px solid #00c6ff;
    border-radius: 15px;
    padding: 15px;
    margin: 5px;
    text-align: center;
}
footer {display: none !important}
"""

with gr.Blocks(css=css, title="🏥 MediBot India") as demo:

    # Header
    gr.HTML("""
    <div style="text-align:center; padding: 30px 0 10px 0;">
        <div style="font-size:3em; margin-bottom:10px;">💊</div>
        <div style="font-size:2.2em; font-weight:900; 
                    background: linear-gradient(90deg, #00c6ff, #0072ff, #00c6ff);
                    -webkit-background-clip: text;
                    -webkit-text-fill-color: transparent;">
            MediBot India
        </div>
        <div style="color:#a0aec0; font-size:1.1em; margin-top:5px;">
            Powered by Mistral 7B • 361,052 Medical Documents • 100% Offline
        </div>
        <div style="margin-top:10px; padding:8px 20px; 
                    background:rgba(255,100,100,0.15); 
                    border:1px solid #ff6464; border-radius:20px;
                    display:inline-block; color:#ff6464; font-size:0.9em;">
            ⚠️ Educational purposes only — Always consult a real doctor!
        </div>
    </div>
    """)

    # Stats Row
    gr.HTML("""
    <div style="display:flex; justify-content:center; gap:15px; margin:20px 0;">
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff; 
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">361K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Medical Documents</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff; 
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">253K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Indian Medicines</div>
        </div>
        <div style="background:rgba(0,198,255,0.1); border:1px solid #00c6ff; 
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#00c6ff;">16K+</div>
            <div style="color:#a0aec0; font-size:0.85em;">Medical Q&As</div>
        </div>
        <div style="background:rgba(118,75,162,0.2); border:1px solid #764ba2; 
                    border-radius:12px; padding:12px 20px; text-align:center;">
            <div style="font-size:1.5em; font-weight:bold; color:#b794f4;">100%</div>
            <div style="color:#a0aec0; font-size:0.85em;">Offline & Private</div>
        </div>
    </div>
    """)

    with gr.Row():
        # Left - Chat
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
              height=450,
             label="Chat with your Medical Assistant",
               avatar_images=(None, "https://api.dicebear.com/7.x/bottts/svg?seed=medibot")
            )
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="💊 Ask about any medicine... e.g. 'What is Paracetamol for?'",
                    label="",
                    scale=5,
                    container=False
                )
                submit_btn = gr.Button("Ask 🚀", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ Clear Chat", size="sm")

        # Right - Info Panel
        with gr.Column(scale=1):
            gr.Markdown("### ⚡ Quick Questions")
            
            q1 = gr.Button("💊 What is Paracetamol for?", size="sm")
            q2 = gr.Button("💰 Price of Augmentin 625?", size="sm")
            q3 = gr.Button("🔬 Medicines with Amoxicillin?", size="sm")
            q4 = gr.Button("⚠️ Side effects of Azithromycin?", size="sm")
            q5 = gr.Button("🤒 Medicines for fever?", size="sm")
            q6 = gr.Button("😴 Medicines for sleep?", size="sm")

            gr.Markdown("---")
            gr.Markdown("""
### 🔧 Tech Stack
🤖 **LLM:** Mistral 7B  
🔍 **Search:** MiniLM Embeddings  
🗄️ **DB:** ChromaDB  
🦜 **Framework:** LangChain  
🖥️ **Runs:** Local GPU (RTX)
            """)

    # Button actions
    submit_btn.click(ask_assistant, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    question_box.submit(ask_assistant, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    clear_btn.click(clear_chat, outputs=[chatbot, question_box])

    # Quick question buttons
    q1.click(lambda h: ask_assistant("What is Paracetamol used for?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q2.click(lambda h: ask_assistant("Price of Augmentin 625?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q3.click(lambda h: ask_assistant("What medicines contain Amoxicillin?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q4.click(lambda h: ask_assistant("Side effects of Azithromycin?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q5.click(lambda h: ask_assistant("What medicines are used for fever in India?", h), inputs=[chatbot], outputs=[chatbot, question_box])
    q6.click(lambda h: ask_assistant("What medicines help with sleep?", h), inputs=[chatbot], outputs=[chatbot, question_box])

demo.launch()